# Deeper GNNs on Elliptic++

Train **GCN, GraphSAGE, and GAT** on two node-classification tasks:
1. **Transactions** — tx ↔ tx money-flow graph from `transactions_data/`.
2. **Wallets** — wallet ↔ wallet `AddrAddr` graph from `actors_data/`.

## Why these data sources for each task
- **Tx task → `transactions_data/`**: it is the only file set with a tx ↔ tx edge list. `actors_data/` only has wallet ↔ tx edges, which can't power a homogeneous tx-graph GNN on its own.
- **Wallet task → `actors_data/`**: wallet features come from `wallets_features.txt` (one row per (address, time-step); we keep the latest row per wallet to capture cumulative state) and edges come from `AddrAddr_edgelist.txt` (direct wallet ↔ wallet interaction).

## Improvements over `initial_experiements.ipynb`
Same three architectures, but with:
- **3 conv layers** instead of 2.
- **Residual connections + LayerNorm + dropout** in every block to combat over-smoothing and stabilise deeper stacks.
- **Wider hidden state** (128 for GCN/SAGE, 32×4 heads for GAT).
- **Early stopping on val illicit-F1** with model checkpoint restore — picks the epoch the model actually generalises best, instead of training to a fixed budget.

## Split (chronological)
Train: time steps **1–29**, Val: **30–34**, Test: **35–49** ("the cliff"). Labels: `1` illicit, `0` licit, `-1` unknown (kept in graph for message passing, excluded from loss/metrics).

In [7]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv, GATConv
from torch_geometric.utils import to_undirected

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "transactions_data").exists():
    ROOT = ROOT.parent
TRANSACTIONS_DIR = ROOT / "transactions_data"
WALLETS_DIR = ROOT / "actors_data"

TRAIN_END = 29
VAL_END = 34
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"root={ROOT}  device={DEVICE}")

root=/Users/jarayliu/Documents/GitHub/stat175-final  device=cpu


## 1. Load both graphs (single cell)

Helper `build_pyg_graph` does the boilerplate (NaN fill → standardise on train rows → build `edge_index` → make undirected → attach masks). We call it twice: once for transactions, once for wallets.

For wallets, multiple `(address, time-step)` rows collapse to **one node per address** by taking the row with the latest time step (which carries the wallet's cumulative state by construction of the dataset).

In [8]:
def build_pyg_graph(node_table, edge_table, node_id_col, edge_cols, feature_cols):
    """node_table needs columns: node_id_col, 'Time step', 'label' (∈ {-1, 0, 1}), and feature_cols."""
    features = node_table[feature_cols].fillna(0).astype(np.float32).values
    time_steps = node_table["Time step"].values
    labels = node_table["label"].values
    is_labelled = labels != -1

    scaler = StandardScaler().fit(features[is_labelled & (time_steps <= TRAIN_END)])
    features = scaler.transform(features).astype(np.float32)

    node_id_to_index = {nid: i for i, nid in enumerate(node_table[node_id_col].values)}
    src = edge_table[edge_cols[0]].map(node_id_to_index)
    dst = edge_table[edge_cols[1]].map(node_id_to_index)
    valid = src.notna() & dst.notna()
    edge_index = torch.tensor(
        np.stack([src[valid].astype(np.int64).values, dst[valid].astype(np.int64).values]),
        dtype=torch.long,
    )
    edge_index = to_undirected(edge_index, num_nodes=len(node_table))

    graph = Data(
        x=torch.from_numpy(features),
        edge_index=edge_index,
        y=torch.tensor(np.where(labels < 0, 0, labels), dtype=torch.long),
    )
    graph.train_mask = torch.tensor(is_labelled & (time_steps <= TRAIN_END))
    graph.val_mask   = torch.tensor(is_labelled & (time_steps > TRAIN_END) & (time_steps <= VAL_END))
    graph.test_mask  = torch.tensor(is_labelled & (time_steps > VAL_END))
    return graph.to(DEVICE)

# === Transactions ===
transactions = pd.read_csv(TRANSACTIONS_DIR / "txs_features.csv")
transaction_classes = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.csv")
transaction_classes["label"] = transaction_classes["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)
transactions = transactions.merge(transaction_classes[["txId", "label"]], on="txId", how="left")
transactions["label"] = transactions["label"].fillna(-1).astype(np.int8)
transaction_feature_cols = [c for c in transactions.columns if c not in ("txId", "Time step", "label")]

transaction_edges = pd.read_csv(TRANSACTIONS_DIR / "txs_edgelist.csv")
transaction_graph = build_pyg_graph(
    transactions, transaction_edges,
    node_id_col="txId", edge_cols=("txId1", "txId2"), feature_cols=transaction_feature_cols,
)

# === Wallet node table ===
with open(WALLETS_DIR / "wallets_features.txt") as f:
    wallet_columns = f.readline().rstrip("\n").split(",")
wallet_dtype = {c: np.float32 for c in wallet_columns if c not in ("address", "Time step")}
wallet_dtype["address"] = "string"
wallet_dtype["Time step"] = np.int16
wallet_features_per_step = pd.read_csv(WALLETS_DIR / "wallets_features.txt", dtype=wallet_dtype)
wallet_classes = pd.read_csv(WALLETS_DIR / "wallets_classes.txt")
wallet_classes["label"] = wallet_classes["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)

# Collapse to one row per wallet by taking the latest time step (cumulative state).
wallets = (wallet_features_per_step
           .sort_values(["address", "Time step"])
           .groupby("address", as_index=False)
           .last())
wallets = wallets.merge(wallet_classes[["address", "label"]], on="address", how="inner")
wallet_feature_cols = [c for c in wallets.columns if c not in ("address", "Time step", "label")]

# === All three wallet edge types ===
addr_addr_edges = pd.read_csv(WALLETS_DIR / "AddrAddr_edgelist.txt")
addr_tx_edges   = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")
tx_addr_edges   = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")

# Homogeneous wallet graph (uses AddrAddr only — for the GCN/SAGE/GAT runs in §5).
wallet_graph = build_pyg_graph(
    wallets, addr_addr_edges,
    node_id_col="address", edge_cols=("input_address", "output_address"), feature_cols=wallet_feature_cols,
)

for name, g in [("Transaction", transaction_graph), ("Wallet (AddrAddr only)", wallet_graph)]:
    print(f"{name:24s}  nodes={g.num_nodes:>9,}  edges={g.num_edges:>9,}  features={g.num_node_features:>3}  "
          f"train={int(g.train_mask.sum()):>7,}  val={int(g.val_mask.sum()):>6,}  test={int(g.test_mask.sum()):>7,}")
print(f"Wallet edge files         AddrAddr={len(addr_addr_edges):>9,}  AddrTx={len(addr_tx_edges):>9,}  TxAddr={len(tx_addr_edges):>9,}")

Transaction               nodes=  203,769  edges=  468,710  features=182  train= 26,381  val= 3,513  test= 16,670
Wallet (AddrAddr only)    nodes=  822,942  edges=5,553,655  features= 55  train=148,038  val=22,366  test= 94,950
Wallet edge files         AddrAddr=2,868,964  AddrTx=  477,117  TxAddr=  837,124


## 2. Models

All three share the same skeleton: input projection → `num_layers` blocks of `[Conv → LayerNorm → activation → dropout → residual]` → linear classifier. The only thing that varies is the `Conv` operator (`GCNConv` / `SAGEConv` / `GATConv`) and the activation. Sharing the skeleton makes the comparison clean.

In [ ]:
class ResidualGNN(nn.Module):
    """3-layer GNN with residual connections + LayerNorm + dropout. Conv operator is pluggable."""
    def __init__(self, conv_factory, input_dim, hidden_dim, output_dim=2, num_layers=3, dropout=0.4, activation=F.relu):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.convs = nn.ModuleList([conv_factory(hidden_dim, hidden_dim) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])
        self.classifier = nn.Linear(hidden_dim, output_dim)
        self.dropout = dropout
        self.activation = activation

    def forward(self, x, edge_index):
        h = self.input_proj(x)
        for conv, norm in zip(self.convs, self.norms):
            residual = h
            h = self.activation(norm(conv(h, edge_index)))
            h = F.dropout(h, p=self.dropout, training=self.training)
            h = h + residual
        return self.classifier(h)

def make_gcn(input_dim, hidden_dim=128):
    return ResidualGNN(GCNConv, input_dim, hidden_dim)

def make_graphsage(input_dim, hidden_dim=128):
    return ResidualGNN(SAGEConv, input_dim, hidden_dim)

def make_gat(input_dim, hidden_per_head=32, heads=4, dropout=0.4):
    # GATConv with concat=True (default) outputs hidden_per_head * heads,
    # so the residual skeleton just needs hidden_dim = hidden_per_head * heads.
    hidden_dim = hidden_per_head * heads
    conv_factory = lambda in_d, out_d: GATConv(in_d, hidden_per_head, heads=heads, dropout=dropout)
    return ResidualGNN(conv_factory, input_dim, hidden_dim, dropout=dropout, activation=F.elu)

## 3. Training loop

Full-batch transductive training. Weighted cross-entropy with `class_weight = [1, n_licit_train / n_illicit_train]`. Each `eval_every` epochs we score val illicit-F1; whenever it improves we snapshot the model state. After early-stop or budget exhaustion we restore the best snapshot before reporting val + test classification reports.

In [ ]:
def train_and_evaluate(model, graph, name, epochs=200, lr=1e-3, weight_decay=5e-4, eval_every=5, patience=30):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    n_pos = int((graph.y[graph.train_mask] == 1).sum())
    n_neg = int((graph.y[graph.train_mask] == 0).sum())
    class_weight = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float, device=DEVICE)

    best_val_f1 = 0.0
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(graph.x, graph.edge_index)
        loss = F.cross_entropy(logits[graph.train_mask], graph.y[graph.train_mask], weight=class_weight)
        loss.backward()
        optimizer.step()

        if epoch % eval_every == 0:
            model.eval()
            with torch.no_grad():
                val_pred = model(graph.x, graph.edge_index)[graph.val_mask].argmax(dim=-1).cpu().numpy()
            val_f1 = f1_score(graph.y[graph.val_mask].cpu().numpy(), val_pred,
                              pos_label=1, zero_division=0)
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += eval_every
            print(f"  epoch {epoch:3d}  loss={loss.item():.4f}  val_illicit_F1={val_f1:.4f}  best={best_val_f1:.4f}")
            if epochs_without_improvement >= patience:
                print(f"  early stop at epoch {epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        predictions = model(graph.x, graph.edge_index).argmax(dim=-1)
    print(f"\n{name} | best val illicit-F1: {best_val_f1:.4f}")
    for split_name, mask in [("VAL  (in-distribution)", graph.val_mask),
                             ("TEST (the cliff)", graph.test_mask)]:
        y_true = graph.y[mask].cpu().numpy()
        y_pred = predictions[mask].cpu().numpy()
        print(f"{split_name}:")
        print(classification_report(y_true, y_pred, target_names=["licit", "illicit"],
                                    digits=4, zero_division=0))
    return model

## 4. Transaction classification

In [11]:
input_dim = transaction_graph.num_node_features
for name, factory in [("GCN", make_gcn), ("GraphSAGE", make_graphsage), ("GAT", make_gat)]:
    print(f"\n=== Transactions · {name} ===")
    train_and_evaluate(factory(input_dim), transaction_graph, f"Transactions · {name}")


=== Transactions · GCN ===
  epoch   5  loss=0.3843  val_illicit_F1=0.4189  best=0.4189
  epoch  10  loss=0.3047  val_illicit_F1=0.5445  best=0.5445
  epoch  15  loss=0.2705  val_illicit_F1=0.5112  best=0.5445
  epoch  20  loss=0.2420  val_illicit_F1=0.5277  best=0.5445
  epoch  25  loss=0.2194  val_illicit_F1=0.5550  best=0.5550
  epoch  30  loss=0.2050  val_illicit_F1=0.5535  best=0.5550
  epoch  35  loss=0.1923  val_illicit_F1=0.5820  best=0.5820
  epoch  40  loss=0.1808  val_illicit_F1=0.6042  best=0.6042
  epoch  45  loss=0.1725  val_illicit_F1=0.5941  best=0.6042
  epoch  50  loss=0.1647  val_illicit_F1=0.6111  best=0.6111
  epoch  55  loss=0.1552  val_illicit_F1=0.6387  best=0.6387
  epoch  60  loss=0.1520  val_illicit_F1=0.6483  best=0.6483
  epoch  65  loss=0.1433  val_illicit_F1=0.6633  best=0.6633
  epoch  70  loss=0.1390  val_illicit_F1=0.6662  best=0.6662
  epoch  75  loss=0.1302  val_illicit_F1=0.6756  best=0.6756
  epoch  80  loss=0.1285  val_illicit_F1=0.6853  best=0.6

## 5. Wallet classification

The wallet graph is much larger (≈820K nodes, ≈5.7M undirected edges after symmetrising `AddrAddr`). Full-batch training is feasible but slower per epoch — if needed, drop `epochs` further or switch to mini-batched neighbour sampling (`NeighborLoader`) later.

In [6]:
input_dim = wallet_graph.num_node_features
for name, factory in [("GCN", make_gcn), ("GraphSAGE", make_graphsage), ("GAT", make_gat)]:
    print(f"\n=== Wallets · {name} ===")
    train_and_evaluate(factory(input_dim), wallet_graph, f"Wallets · {name}", epochs=100)


=== Wallets · GCN ===
  epoch   5  loss=0.3511  val_illicit_F1=0.2708  best=0.2708
  epoch  10  loss=0.2703  val_illicit_F1=0.3039  best=0.3039
  epoch  15  loss=0.2423  val_illicit_F1=0.3121  best=0.3121
  epoch  20  loss=0.2207  val_illicit_F1=0.3157  best=0.3157
  epoch  25  loss=0.2064  val_illicit_F1=0.3379  best=0.3379
  epoch  30  loss=0.1970  val_illicit_F1=0.3161  best=0.3379
  epoch  35  loss=0.1891  val_illicit_F1=0.3232  best=0.3379


KeyboardInterrupt: 

## 6. Heterogeneous wallet graph (uses all three edge types)

The wallet GNNs above only saw `AddrAddr` — direct wallet ↔ wallet co-spending. The richer signal lives in `AddrTx ∘ TxAddr`: wallet → tx → wallet money-flow paths through explicit transaction nodes, which `AddrAddr` alone cannot express.

We build a `HeteroData` graph with two node types:
- **`wallet`** (≈823K nodes, 55 features)
- **`tx`** (≈204K nodes, 182 features)

and three forward relations (each made bidirectional via `ToUndirected`):
- `(wallet, addr_to_addr, wallet)` — `AddrAddr_edgelist`
- `(wallet, addr_to_tx,   tx)`     — `AddrTx_edgelist`
- `(tx,     tx_to_addr,   wallet)` — `TxAddr_edgelist`

The model is a 2-layer **R-GCN-style** network (`HeteroConv` over the six relations: 3 forward + 3 reverse). Each layer runs an independent `SAGEConv` per relation and **sums** the per-relation messages at every node. Loss is computed only on labelled wallet nodes — the tx node embeddings are learned implicitly to support the wallet predictions.

In [ ]:
from torch_geometric.data import HeteroData
from torch_geometric.transforms import ToUndirected

# Index maps for the two node types
wallet_to_idx = {addr: i for i, addr in enumerate(wallets["address"].values)}
tx_to_idx     = {tx: i for i, tx in enumerate(transactions["txId"].values)}

# Standardise wallet features on labelled training rows only
wallet_feat = wallets[wallet_feature_cols].fillna(0).astype(np.float32).values
wallet_ts   = wallets["Time step"].values
wallet_y    = wallets["label"].values
wallet_lab  = wallet_y != -1
wallet_feat = StandardScaler().fit(wallet_feat[wallet_lab & (wallet_ts <= TRAIN_END)]).transform(wallet_feat).astype(np.float32)

# Standardise tx features on labelled training rows only
tx_feat = transactions[transaction_feature_cols].fillna(0).astype(np.float32).values
tx_ts   = transactions["Time step"].values
tx_y    = transactions["label"].values
tx_lab  = tx_y != -1
tx_feat = StandardScaler().fit(tx_feat[tx_lab & (tx_ts <= TRAIN_END)]).transform(tx_feat).astype(np.float32)

def map_edges(edge_df, src_col, dst_col, src_map, dst_map):
    src = edge_df[src_col].map(src_map)
    dst = edge_df[dst_col].map(dst_map)
    valid = src.notna() & dst.notna()
    return torch.tensor(
        np.stack([src[valid].astype(np.int64).values, dst[valid].astype(np.int64).values]),
        dtype=torch.long,
    )

hetero_graph = HeteroData()
hetero_graph["wallet"].x = torch.from_numpy(wallet_feat)
hetero_graph["tx"].x     = torch.from_numpy(tx_feat)
hetero_graph["wallet", "addr_to_addr", "wallet"].edge_index = map_edges(addr_addr_edges, "input_address", "output_address", wallet_to_idx, wallet_to_idx)
hetero_graph["wallet", "addr_to_tx",   "tx"    ].edge_index = map_edges(addr_tx_edges,   "input_address", "txId",           wallet_to_idx, tx_to_idx)
hetero_graph["tx",     "tx_to_addr",   "wallet"].edge_index = map_edges(tx_addr_edges,   "txId",          "output_address", tx_to_idx,     wallet_to_idx)

# Wallet labels and chronological masks (we only classify wallets)
hetero_graph["wallet"].y          = torch.tensor(np.where(wallet_y < 0, 0, wallet_y), dtype=torch.long)
hetero_graph["wallet"].train_mask = torch.tensor(wallet_lab & (wallet_ts <= TRAIN_END))
hetero_graph["wallet"].val_mask   = torch.tensor(wallet_lab & (wallet_ts > TRAIN_END) & (wallet_ts <= VAL_END))
hetero_graph["wallet"].test_mask  = torch.tensor(wallet_lab & (wallet_ts > VAL_END))

# Add the reverse of every relation so messages flow in both directions
hetero_graph = ToUndirected()(hetero_graph)
hetero_graph = hetero_graph.to(DEVICE)

print("Node types:")
for ntype in hetero_graph.node_types:
    print(f"  {ntype:6s} nodes={hetero_graph[ntype].num_nodes:>9,}  features={hetero_graph[ntype].x.shape[1]}")
print("Edge types (after ToUndirected):")
for etype in hetero_graph.edge_types:
    print(f"  {etype}  edges={hetero_graph[etype].edge_index.shape[1]:>9,}")
print(f"Wallet labels  train={int(hetero_graph['wallet'].train_mask.sum()):,}  "
      f"val={int(hetero_graph['wallet'].val_mask.sum()):,}  test={int(hetero_graph['wallet'].test_mask.sum()):,}")

In [ ]:
from torch_geometric.nn import HeteroConv

class HeteroSAGE(nn.Module):
    """2-layer heterogeneous GNN: per-relation SAGEConv, summed across relations at each node."""
    def __init__(self, edge_types, hidden_dim=64, dropout=0.4):
        super().__init__()
        self.convs = nn.ModuleList([
            HeteroConv({et: SAGEConv((-1, -1), hidden_dim) for et in edge_types}, aggr="sum"),
            HeteroConv({et: SAGEConv((-1, -1), hidden_dim) for et in edge_types}, aggr="sum"),
        ])
        self.norm = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 2)
        self.dropout = dropout

    def forward(self, x_dict, edge_index_dict):
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {k: F.dropout(F.relu(v), p=self.dropout, training=self.training)
                      for k, v in x_dict.items()}
        return self.classifier(self.norm(x_dict["wallet"]))


def train_and_evaluate_hetero(model, hetero_graph, name, epochs=50, lr=1e-3,
                              weight_decay=5e-4, eval_every=2, patience=10):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_mask = hetero_graph["wallet"].train_mask
    val_mask   = hetero_graph["wallet"].val_mask
    test_mask  = hetero_graph["wallet"].test_mask
    y          = hetero_graph["wallet"].y

    n_pos = int((y[train_mask] == 1).sum())
    n_neg = int((y[train_mask] == 0).sum())
    class_weight = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float, device=DEVICE)

    best_val_f1 = 0.0
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(hetero_graph.x_dict, hetero_graph.edge_index_dict)
        loss = F.cross_entropy(logits[train_mask], y[train_mask], weight=class_weight)
        loss.backward()
        optimizer.step()

        if epoch % eval_every == 0:
            model.eval()
            with torch.no_grad():
                val_pred = (model(hetero_graph.x_dict, hetero_graph.edge_index_dict)[val_mask]
                            .argmax(dim=-1).cpu().numpy())
            val_f1 = f1_score(y[val_mask].cpu().numpy(), val_pred, pos_label=1, zero_division=0)
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += eval_every
            print(f"  epoch {epoch:3d}  loss={loss.item():.4f}  val_illicit_F1={val_f1:.4f}  best={best_val_f1:.4f}")
            if epochs_without_improvement >= patience:
                print(f"  early stop at epoch {epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        predictions = model(hetero_graph.x_dict, hetero_graph.edge_index_dict).argmax(dim=-1)
    print(f"\n{name} | best val illicit-F1: {best_val_f1:.4f}")
    for split_name, mask in [("VAL  (in-distribution)", val_mask),
                             ("TEST (the cliff)", test_mask)]:
        y_true = y[mask].cpu().numpy()
        y_pred = predictions[mask].cpu().numpy()
        print(f"{split_name}:")
        print(classification_report(y_true, y_pred, target_names=["licit", "illicit"],
                                    digits=4, zero_division=0))
    return model


print("=== Wallets · Hetero SAGE (all 3 edge types) ===")
train_and_evaluate_hetero(
    HeteroSAGE(edge_types=hetero_graph.edge_types, hidden_dim=64),
    hetero_graph, "Wallets · Hetero SAGE",
)

## Where to go next
- **Heterogeneous wallet+tx graph** for the wallet task — `HeteroData` over `AddrAddr`, `AddrTx`, `TxAddr` with R-GCN / HAN / HGT.
- **Neighbour sampling** (`NeighborLoader`) so the wallet GNN scales beyond full-batch and lets us push more epochs / deeper stacks.
- **Threshold calibration on val** — pick the decision threshold that maximises val illicit-F1 instead of fixing it at the 0.5 implied by `argmax`.
- **Per-time-step test breakdown** to localise the cliff (most models lose almost all illicit recall around step 43).